In [1]:
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix 
from tqdm import tqdm
import numpy as np

In [2]:
def load_data(file_path:str, target_column:str, test_size:float=0.2, val_size:float=0.2, random_state:int=42):
    data = pd.read_csv(file_path)
    X = data.drop(target_column, axis=1)
    y = data[target_column]
    
    # scale X
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=random_state)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_size, random_state=random_state)

    return X_train, X_val, X_test, y_train, y_val, y_test, scaler


In [7]:
def best_svm_classifier(X_train, X_val, X_test, y_train, y_val, y_test, verbose=False):
    # Define parameter grid
    param_grid = {
        'C': [0.1, 1, 10, 100, 1000],
        'gamma': [0.1, 1, 10, 100, 1000],
        'kernel': ['linear', 'rbf', 'sigmoid']
    }
    
    # Create SVM classifier
    svm = SVC()
    
    # Create GridSearchCV object
    grid_search = GridSearchCV(
        estimator=svm,
        param_grid=param_grid,
        cv=5,
        n_jobs=-1,
        verbose=2 if verbose else 0,
        scoring='accuracy'
    )
    
    # Fit the model
    grid_search.fit(X_train, y_train)
    
    if verbose:
        print(f"Best parameters: {grid_search.best_params_}")
        print(f"Best cross-validation score: {grid_search.best_score_:.3f}")
    
    # Validate on validation set
    val_score = grid_search.score(X_val, y_val)

    grid_predictions = grid_search.predict(X_test) 
    
    # print classification report 
    print(classification_report(y_test, grid_predictions)) 
    
    return (grid_search.best_estimator_, 
            val_score,
            grid_search.best_params_['C'],
            grid_search.best_params_['gamma'])

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test, scaler = load_data('data/pima_indians.csv', 'Outcome', random_state=42)
# COMMENT
best_model, best_score, best_c, best_gamma = best_svm_classifier(X_train, X_val, X_test, y_train, y_val, y_test, verbose=True)

Fitting 5 folds for each of 75 candidates, totalling 375 fits
Best parameters: {'C': 1, 'gamma': 0.1, 'kernel': 'rbf'}
Best cross-validation score: 0.758
              precision    recall  f1-score   support

           0       0.77      0.84      0.80        99
           1       0.65      0.55      0.59        55

    accuracy                           0.73       154
   macro avg       0.71      0.69      0.70       154
weighted avg       0.73      0.73      0.73       154

